# Veritas Lens — Fine-tuning DeBERTa-v3 on WELFake

Headline notebook for the NeuroLogic '26 fake-news track. Runs on a free Colab T4 in ~5 hours total (one fold ≈ 55 minutes).

**Steps**
1. Load and split WELFake (5-fold stratified).
2. Fine-tune `microsoft/deberta-v3-base` per fold with the same recipe as `ml/scripts/finetune_deberta.py`.
3. Soft-vote across folds, fit isotonic calibration.
4. Run the L-Defense rationale prompt on a sample of test articles.
5. Run the LLM-paraphrase robustness audit on the held-out set.

Outputs land in `ml/checkpoints/deberta-v3/` and feed the leaderboard JSON exported by `ml/scripts/export_metrics.py`.

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
import os, json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.isotonic import IsotonicRegression
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset

DATA = '../data/welfake.csv'  # columns: title, body, label (0=REAL, 1=FAKE)
df = pd.read_csv(DATA)
print(df.shape, df['label'].value_counts(normalize=True).to_dict())

In [ ]:
MODEL = 'microsoft/deberta-v3-base'
MAX_LEN = 384
BATCH = 16
EPOCHS = 3
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def make_text(row):
    return (row['title'] or '') + ' [SEP] ' + (row['body'] or '')

df['text'] = df.apply(make_text, axis=1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(df))
fold_metrics = []

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)


In [ ]:
for fold, (tr, va) in enumerate(skf.split(df['text'], df['label'])):
    print(f'--- fold {fold} ---')
    train_ds = Dataset.from_pandas(df.iloc[tr][['text','label']]).map(tok, batched=True)
    val_ds   = Dataset.from_pandas(df.iloc[va][['text','label']]).map(tok, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    args = TrainingArguments(
        output_dir=f'../checkpoints/deberta-v3/fold{fold}',
        learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.06,
        lr_scheduler_type='cosine', per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH*2, num_train_epochs=EPOCHS,
        fp16=torch.cuda.is_available(), evaluation_strategy='epoch',
        save_strategy='epoch', save_total_limit=1, load_best_model_at_end=True,
        metric_for_best_model='eval_loss', greater_is_better=False,
        report_to=[], seed=42+fold)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorWithPadding(tokenizer), tokenizer=tokenizer)
    trainer.train()
    out = trainer.predict(val_ds)
    proba = torch.softmax(torch.from_numpy(out.predictions), dim=-1)[:,1].numpy()
    oof[va] = proba
    pred = (proba >= 0.5).astype(int)
    fold_metrics.append({'fold': fold,
                          'accuracy': float(accuracy_score(df['label'].iloc[va], pred)),
                          'f1': float(f1_score(df['label'].iloc[va], pred)),
                          'roc_auc': float(roc_auc_score(df['label'].iloc[va], proba))})
    print(fold_metrics[-1])


In [ ]:
y = df['label'].to_numpy()
pred = (oof >= 0.5).astype(int)
print('OOF accuracy:', accuracy_score(y, pred))
print('OOF F1:', f1_score(y, pred))
print('OOF ROC-AUC:', roc_auc_score(y, oof))
print(classification_report(y, pred, target_names=['REAL','FAKE']))
print(confusion_matrix(y, pred))

iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y)
import joblib
os.makedirs('../checkpoints/deberta-v3', exist_ok=True)
joblib.dump(iso, '../checkpoints/deberta-v3/isotonic.joblib')
with open('../checkpoints/deberta-v3/fold_metrics.json','w') as f:
    json.dump(fold_metrics, f, indent=2)


## L-Defense rationale (post-hoc)

Following Liu et al. (WebConf 2024). We prompt Llama 3.3 70B via Groq to argue both sides and return a JSON debate. Identical prompt to the web app's `/api/rationale` route.

In [ ]:
import os, requests, json
GROQ_KEY = os.environ.get('GROQ_API_KEY', '')
PROMPT = '''You are an evidence-based fact-checking analyst. Argue both sides of the article (REAL vs FAKE) following L-Defense, then commit to a verdict. Return JSON only with keys: summary, realArguments[{point, evidence}], fakeArguments[{point, evidence}].'''

def rationale(title, body, verdict, p):
    if not GROQ_KEY:
        return {'summary': 'No Groq key — fall back to local engine.'}
    r = requests.post('https://api.groq.com/openai/v1/chat/completions',
        headers={'Authorization': f'Bearer {GROQ_KEY}', 'Content-Type': 'application/json'},
        json={'model': 'llama-3.3-70b-versatile', 'temperature': 0.2,
              'response_format': {'type': 'json_object'},
              'messages': [{'role':'system', 'content': PROMPT},
                           {'role':'user', 'content': f'Title: {title}\nBody: {body}\nVerdict: {verdict} (P={p:.3f})'}]},
        timeout=60).json()
    return json.loads(r['choices'][0]['message']['content'])

sample = df.sample(3, random_state=0)
for _, row in sample.iterrows():
    out = rationale(row['title'], row['body'], 'FAKE' if row['label']==1 else 'REAL', 0.95)
    print(out)
